# Predictive Paradox — Power Grid Demand Forecasting

> **Task:** Predict the next hour's electricity demand (`demand_mw`) on the Bangladesh national grid using only information available at the current time.

## Pipeline Overview

| Stage | Module | Key Decisions |
|-------|--------|---------------|
| Data Loading | `src/data_loader.py` | Parse timestamps, fix weather header |
| Preprocessing | `src/preprocessing.py` | De-dup → hourly reindex → IQR outlier filter → merge |
| Feature Engineering | `src/feature_engineering.py` | Cyclical calendar + lags + rolling stats |
| Modelling | `src/model.py` | XGBoost & LightGBM, strict chronological split |

**Constraint:** No LSTMs, Transformers, ARIMA, or Prophet — classical ML only.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import sys

# Add src/ to path so we can import our modules
sys.path.append(str(Path('.').resolve()))

from src.data_loader         import load_all
from src.preprocessing       import run_preprocessing, rolling_iqr_clean, clean_pgcb
from src.feature_engineering import build_features
from src.model               import chronological_split, train_and_evaluate, print_summary
from src.model               import plot_actual_vs_predicted, plot_error_analysis
from src.model               import plot_feature_importance, plot_mape_by_hour

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False
})

DATA_DIR = Path('./data')
FIG_DIR  = Path('./figures')
FIG_DIR.mkdir(exist_ok=True)

print('Environment ready ✓')

---
## 1. Data Loading

We load three separate data sources that operate at different temporal granularities:

- **PGCB** — hourly grid telemetry (our target variable lives here)
- **Weather** — hourly environmental readings (the raw file has a 4-row preamble that we skip)
- **Economic** — annual World Bank macro-indicators (broadcast to hourly by year)

In [ ]:
df_pgcb, df_weather, df_econ = load_all(DATA_DIR)

In [ ]:
print("── PGCB columns ──────────────────────────────")
print(df_pgcb.dtypes)
df_pgcb.head(3)

In [ ]:
print("── Weather sample ────────────────────────────")
df_weather.describe().round(2)

In [ ]:
print("── Economic indicators (annual) ──────────────")
df_econ.round(2)

---
## 2. Exploratory Data Analysis (EDA)

Before cleaning, we visualise the raw demand signal to understand the extent of noise and identify structural patterns.

In [ ]:
# ── 2a. Raw demand time series ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 3))
ts = df_pgcb.set_index('datetime').sort_index()['demand_mw']
ax.plot(ts.index, ts.values, lw=0.4, color='steelblue')
ax.set_title('Raw demand_mw — full history (note spikes)', fontweight='bold')
ax.set_ylabel('MW')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_raw_demand.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2b. Distribution of demand_mw ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(ts.dropna(), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Demand distribution (raw)', fontweight='bold')
axes[0].set_xlabel('MW')

monthly = ts.resample('M').mean()
axes[1].plot(monthly.index, monthly.values, color='darkorange', lw=2)
axes[1].set_title('Monthly average demand', fontweight='bold')
axes[1].set_ylabel('MW')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2c. Average hourly profile (seasonality check) ─────────────────────────
ts_df = ts.to_frame(name='demand').dropna()
ts_df['hour'] = ts_df.index.hour
ts_df['month'] = ts_df.index.month

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
ts_df.groupby('hour')['demand'].mean().plot(ax=axes[0], color='steelblue', marker='o', ms=4)
axes[0].set_title('Average demand by hour of day', fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('MW')

ts_df.groupby('month')['demand'].mean().plot(ax=axes[1], color='darkorange', marker='o', ms=4)
axes[1].set_title('Average demand by month', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('MW')
axes[1].set_xticks(range(1, 13))
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_seasonality.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2d. Correlation heatmap (weather vs demand, one sample year) ───────────
sample = df_pgcb.set_index('datetime').join(df_weather, how='inner')
sample = sample[sample.index.year == 2022][['demand_mw','temperature','humidity',
                                             'apparent_temp','precipitation','cloud_cover']].dropna()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(sample.corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Correlation — demand vs weather (2022 sample)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_correlation.png', bbox_inches='tight')
plt.show()

**EDA Findings:**
- A clear daily cycle with peak demand in the evening (~18:00–22:00)
- A seasonal pattern: demand peaks in summer months (May–August) driven by cooling load
- Temperature has a positive correlation with demand
- Visible extreme spikes in the raw series — these are telemetry errors to be removed

---
## 3. Data Preprocessing

### Strategy

| Problem | Solution | Rationale |
|---------|----------|-----------|
| Duplicate timestamps | Keep first occurrence | Grid telemetry can log the same interval twice |
| Missing hours | Reindex + ffill (≤2 h) | Short telemetry dropouts; longer gaps stay NaN |
| Extreme spikes | Rolling IQR filter (k=4.0) | Conservative — only removes genuine outliers, not real peaks |
| Missing weather | Linear interpolation (≤6 h) | Sensor readings change smoothly |
| Annual econ data | Broadcast by year | No sub-annual variation in macro indicators |

In [ ]:
df_master = run_preprocessing(df_pgcb, df_weather, df_econ)

In [ ]:
# Visualise the effect of the outlier filter
df_raw_ts = df_pgcb.set_index('datetime').sort_index()['demand_mw']

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(df_raw_ts.index, df_raw_ts.values, lw=0.4, color='steelblue')
axes[0].set_title('Raw demand_mw (with telemetry spikes)', fontweight='bold')
axes[1].plot(df_master.index, df_master['demand_mw'].values, lw=0.4, color='darkorange')
axes[1].set_title('Cleaned demand_mw (Rolling IQR, window=72h, k=4.0)', fontweight='bold')
for ax in axes:
    ax.set_ylabel('MW')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_raw_vs_clean.png', bbox_inches='tight')
plt.show()

In [ ]:
print("Master DataFrame info:")
print(f"  Shape   : {df_master.shape}")
print(f"  Date range: {df_master.index.min().date()} → {df_master.index.max().date()}")
print(f"\nMissing values per column:")
missing = df_master.isna().sum()
print(missing[missing > 0].to_string())

---
## 4. Feature Engineering

Because tree-based models have no built-in notion of time (each row is independent), we must manually encode temporal context as features.

### Feature categories

| Category | Examples | Purpose |
|----------|----------|---------|
| **Cyclical calendar** | `hour_sin`, `hour_cos`, `month_sin` | Preserve wrap-around: hour 23 ≈ hour 0 |
| **Binary flags** | `is_weekend`, `is_friday` | Day-type demand patterns |
| **Direct lags** | `lag_1h`, `lag_24h`, `lag_168h` | Short/medium/weekly memory |
| **Rolling stats** | `roll_mean_24h`, `roll_std_6h` | Trend and volatility context |
| **Differentials** | `demand_diff_1h`, `demand_diff_24h` | Rate-of-change signal |
| **Weather** | `temperature`, `humidity`, `cloud_cover` | Environmental drivers |
| **Economic** | `econ_gdp_growth`, `econ_population` | Long-term demand baseline |

In [ ]:
df_feat, features = build_features(df_master)
print(f"Feature matrix : {df_feat.shape}")
print(f"Total features : {len(features)}")

In [ ]:
# Show a few example rows to confirm structure
df_feat[['demand_mw','lag_1h','lag_24h','lag_168h','roll_mean_24h',
         'hour_sin','hour_cos','temperature','target']].tail(5).round(2)

---
## 5. Train / Test Split (Strict Chronological)

We use **all data before 2023** for training and **the entire year 2023** as a hold-out test set.  
This guarantees zero leakage — the model never sees future data during training.

> ⚠️ A random split would be **wrong** here: rolling/lag features computed from future rows would leak information about the test period into training.

In [ ]:
X_train, y_train, X_test, y_test, train, test = chronological_split(df_feat, features)

---
## 6. Model Training & Evaluation

We train two gradient-boosting models and compare on MAPE (primary), MAE, and RMSE.

### Why LightGBM and XGBoost?
Both are ensemble methods that build many shallow decision trees sequentially, each correcting the residual error of the previous.  They handle tabular time-series features natively, require no feature scaling, and are robust to the moderate feature correlations introduced by lag engineering.  LightGBM typically trains faster on larger datasets due to its histogram-based split-finding.

In [ ]:
results = train_and_evaluate(X_train, y_train, X_test, y_test)
print_summary(results)

---
## 7. Visualisations & Interpretation

In [ ]:
best_model = results['LightGBM']['model']
best_preds = results['LightGBM']['preds']

# ── 7a. Actual vs Predicted (first 2 weeks of test set) ────────────────────
plot_actual_vs_predicted(test, best_preds, n_days=14,
                         save_path=FIG_DIR / 'fig_pred_vs_actual.png')

In [ ]:
# ── 7b. Error distribution + scatter ───────────────────────────────────────
plot_error_analysis(y_test, best_preds, save_path=FIG_DIR / 'fig_error_dist.png')

In [ ]:
# ── 7c. Feature importance ──────────────────────────────────────────────────
plot_feature_importance(best_model, features, save_path=FIG_DIR / 'fig_feature_importance.png')

**Feature Importance Interpretation:**
- **Lag features dominate** (especially `lag_1h`, `lag_2h`, `lag_24h`) — the single best predictor of next-hour demand is the current hour's demand. This confirms strong short-range autocorrelation.
- **Rolling statistics** (`roll_mean_24h`, `roll_mean_168h`) capture the recent trend and the weekly baseline respectively.
- **Calendar features** (`hour_sin/cos`) rank highly, reflecting the strong intra-day cycle seen in the EDA.
- **Temperature** is the most important weather variable, consistent with cooling-driven demand in hot months.
- **Economic features** contribute modestly — they shift the long-run baseline but are constant within a year.

In [ ]:
# ── 7d. MAPE by hour of day ─────────────────────────────────────────────────
plot_mape_by_hour(test, best_preds, save_path=FIG_DIR / 'fig_mape_by_hour.png')

---
## 8. Summary & Conclusions

### Results

| Model | MAPE (%) | MAE (MW) | RMSE (MW) |
|-------|----------|----------|-----------|
| LightGBM (primary) | **see output above** | — | — |
| XGBoost | — | — | — |

### Key Design Choices

1. **Rolling IQR at k=4.0** — A deliberate choice to be conservative.  Using a smaller k (e.g. 1.5) would flag genuine peak-load hours as anomalies and degrade the model's ability to learn extreme-demand patterns.

2. **Lag-168h (1 week)** — Weekly periodicity is strong in grid data; the same hour of the same weekday consistently drives similar demand.  This single feature captures most of the inter-day variance.

3. **No normalisation** — Tree-based models are invariant to monotonic feature transformations.  Adding StandardScaler would not help and would obscure the physical interpretation of the features.

4. **Strict chronological split** — Rolling/lag features constructed from future rows would leak future information into training.  The hard year boundary at 2023 is the only safe approach.

### Limitations & Next Steps
- The model does not account for public holidays (Eid, etc.) which cause sharp demand dips
- Load-shedding events in the training data distort the actual-demand signal
- A stacking ensemble of XGBoost + LightGBM may reduce error by 0.2–0.5% MAPE